[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/blob/main/prep/P3_tracking_month.ipynb)


# P3 — ติดตามพายุทั้งเดือนแบบ "เหตุการณ์พายุ" (T-DaTing 15 / 10 / 5 นาที) ตารางเซลล์ เส้นทาง footprint และกริดสรุป

**สำหรับผู้ดูแลข้อมูล** — ต้องรัน P1 และ P2 ก่อน (ไม่อ่าน UF แล้ว ใช้เฉพาะไฟล์ derived)

ใช้ฟังก์ชันของ repo3 (`track_tools`) และพารามิเตอร์ T-DaTing เดิมทุกค่า (`minref=33, maxref=45, mindiff=5, minsize=15, minmax=40, mindis=8, mintrack=2, match_frac=0.25`)

### ทำไมไม่ติดตามทีละวัน
ไฟล์ CAPPI แยกรายวัน 00:00–23:45 UTC ถ้าติดตามทีละไฟล์ พายุที่ยังอยู่ตอน **00 UTC (07:00 น.)** จะถูกหั่นเป็นสองเส้น: อายุและระยะทางสั้นลง จำนวนเส้นทางเกินจริง เกิดจุดกำเนิดปลอม และเส้นสั้นที่คร่อมรอยต่อหายไปเลย (เหลือภาพเดียวในแต่ละวัน) ปฏิทินพายุจาก P1 แสดงว่าหลายคืนมีพายุต่อเนื่องข้ามรอยต่อนี้ โดยเฉพาะช่วงต้นฤดูฝน

### วิธีของ P3 (`utils/tracking.py`)
1. ต่ออนุกรม 15 นาทีทั้งเดือน แล้วแบ่งเป็น **เหตุการณ์ (episode)** เฉพาะที่ **ช่วงสงบ** (พื้นที่ ≥ 35 dBZ < 20 km² ติดกัน ≥ 60 นาที) หรือ **ข้อมูลขาด** — ไม่ตัดที่เที่ยงคืน
2. ติดตามทีละเหตุการณ์ (ขนาน) ภาพ 5/10 นาที (advection interpolation แบบ linZ) สร้างต่อเนื่องทั้งเหตุการณ์ ตัวแปร P2 อ่านจากไฟล์หลายวันต่อกัน
3. เขียน label cube กลับเป็น **ไฟล์รายวัน** (บทเรียนนิสิตใช้ได้ตามเดิม)

ทดสอบแล้ว: เลื่อนเวลา 3 พ.ค. ให้ช่วงพายุตอน 12 UTC กลายเป็นเที่ยงคืน ได้เส้นทาง **ตรงกับการติดตามต่อเนื่องทุกเส้น** (164/164) ส่วนวิธีรายวันเดิมได้ 161 เส้น (เส้นคร่อมรอยต่อถูกหั่นหรือหายไป)

### รหัสและป้ายกำกับ
- `track_id` = episode × 10000 + ลำดับ (ไม่ซ้ำทั้งเดือน, ค่าใน label = track_id + 1), `uid` = `E###-####`
- ตารางเส้นทาง: `day` = วันที่ UTC ของจุดเริ่ม, `day_local` = วันที่เวลาไทย, `episode`, `crosses_00utc`, `crosses_local_midnight`
- ตารางเซลล์: `day` / `day_local` = วันที่ของเวลานั้น
- `censored_start` / `censored_end` = แตะขอบเหตุการณ์ที่ติด **ข้อมูลขาดหรือขอบเดือน** (อายุจริงอาจยาวกว่า) · `at_episode_edge` = แตะขอบเหตุการณ์ไม่ว่าด้วยเหตุใด

| ผลลัพธ์ | ที่เก็บ |
|---|---|
| `tracks/KKN_YYYYMMDD_labels_<tag>.nc` (label cube รายวัน) | Drive → Zenodo |
| `cells_<tag>.csv.gz`, `tracks_<tag>.csv`, `footprints_<tag>.gpkg`, `episodes.csv`, `tracking_summary.csv` | `data/derived/` (ทั้งเดือน) |
| `terrain_1km.nc`, `monthly_summary_1km.nc` (ความถี่ echo, ความหนาแน่นเส้นทาง, จุดกำเนิด, ลูกเห็บ) | `data/derived/` |

**Runtime:** CPU + High-RAM — ราว 1 ชั่วโมงทั้งเดือนบน 8 core รันซ้ำได้ (ผลของแต่ละเหตุการณ์เก็บไว้ใน `tracks/episodes/` ข้ามเหตุการณ์ที่เสร็จแล้ว)

In [ ]:
# @title ⚙️ เตรียมสภาพแวดล้อม (รันก่อนเสมอ)
import os, sys, subprocess
REPO = "Teach_stormTracks_KKN_May2024_HID_doppler"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/nattaponm/{REPO}.git", f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"])
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysteps==1.21.5", "opencv-python-headless"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "prep"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("โฟลเดอร์ทำงาน:", os.getcwd(), "| CPU", os.cpu_count(), "core")

In [ ]:
import json, time, shutil, warnings
from pathlib import Path
from functools import partial
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

from utils.config import load_settings, repo_path
from utils import maps as M
s = load_settings()
M.setup_style()
OUT_ROOT = Path(os.environ.get("KKN_OUT_ROOT", s["radar_data"]["drive_derived_root"]))
N_WORKERS = os.cpu_count()
CTX = mproc.get_context("fork")

# ดึงผลของขั้นก่อนหน้าจาก Drive (Colab แยกเครื่องต่อ notebook — ดู utils/sync.py)
from utils import sync
sync.pull(OUT_ROOT)
s = load_settings()      # อ่านใหม่หลัง pull (พิกัดเรดาร์จริงจาก P0)

import pickle
from utils import tracking as TR, track_tools as tt, data_io, derived_io as DIO, geo as G
CAPPI_DIR, POLAR_DIR, TRACK_DIR = OUT_ROOT / "cappi", OUT_ROOT / "polar", OUT_ROOT / "tracks"
EP_DIR, PART_DIR = TRACK_DIR / "episodes", TRACK_DIR / "parts"
for d in (TRACK_DIR, EP_DIR, PART_DIR):
    d.mkdir(parents=True, exist_ok=True)
DER = repo_path("data", "derived")
TAGS = {k: TR.MAIN_TAGS[k] for k in s["tracking"]["tags"]}
TRK = s["tracking"]
OVERWRITE = False       # True = ติดตามใหม่ทุกเหตุการณ์
cappi_files = sorted(CAPPI_DIR.glob("KKN_*_cappi_15min.nc"))
days = [p.name[4:12] for p in cappi_files]
print(f"{len(days)} วัน: {days[0]} … {days[-1]}")

## 1. กริดภูมิประเทศ (Copernicus GLO-90 → 1 km) และความสูงลำคลื่น

สูตรเดียวกับ repo3: elevation, slope, relief (ช่วงสูงต่ำในหน้าต่าง 11 km), $\partial h/\partial x$, $\partial h/\partial y$, TPI (ความสูงเทียบค่าเฉลี่ย 11 km)

In [ ]:
_, meta = data_io.load_pcappi(str(cappi_files[0]))
dem_path = repo_path(s["dem"]["out"])
if not dem_path.exists():
    G.build_dem(s, out=dem_path)
TERR = TR.terrain_from_dem(dem_path, meta)
BEAM = tt.beam_height_grid(meta)
DIO.write_grids_2d(DER / "terrain_1km.nc", {**TERR, "beam_h0_km": BEAM}, meta,
                   {"source": "Copernicus GLO-90 averaged to 1 km (NB01); formulas as repo3 make_terrain_grid",
                    "beam_elevation_deg": tt.ELEV0})
print({k: f"{np.nanmin(v):.1f}–{np.nanmax(v):.1f}" for k, v in TERR.items()})

## 2. สำรวจทั้งเดือน → แบ่งเป็นเหตุการณ์พายุ

อ่าน CAPPI ทุกวัน (ขนาน) เพื่อหาพื้นที่ ≥ 35 dBZ ต่อภาพ ความถี่ราย pixel และนับลูกเห็บจาก P2 แล้วแบ่งเหตุการณ์ที่ช่วงสงบ ≥ 60 นาทีหรือข้อมูลขาด

In [ ]:
def _scan(d):
    return d, TR.scan_day(CAPPI_DIR / f"KKN_{d}_cappi_15min.nc", POLAR_DIR / f"KKN_{d}_polar_15min.nc", TRK["thr_dbz"])

with ProcessPoolExecutor(N_WORKERS, mp_context=CTX) as ex:
    SCAN = dict(ex.map(_scan, days))
times, avail, area = TR.month_series(SCAN)
EP = TR.find_episodes(times, avail, area, thr_area=TRK["min_area_km2"],
                      quiet_frames=int(TRK["episode_quiet_min"] // 15), pad=TRK["episode_pad_frames"])
EP.to_csv(DER / "episodes.csv", index=False)
print(f"เหตุการณ์ {len(EP)} ครั้ง | ยาวมัธยฐาน {EP['hours'].median():.1f} ชม. | ยาวสุด {EP['hours'].max():.1f} ชม. | "
      f"คร่อม 00 UTC {EP['crosses_00utc'].sum()} ครั้ง")
print("ขอบเหตุการณ์ (start/end):", EP["start_edge"].value_counts().to_dict(), "/", EP["end_edge"].value_counts().to_dict())
EP.sort_values("hours", ascending=False).head(8)

In [ ]:
# ภาพเหตุการณ์บนปฏิทิน: แถบแนวนอน = หนึ่งเหตุการณ์ (สีสลับเพื่อแยกเหตุการณ์ติดกัน)
fig, ax = plt.subplots(figsize=(7.2, 5.4))
dd = (times.normalize() - times[0].normalize()).days.values + 1
hh = times.hour.values + times.minute.values / 60
A = np.where(avail, np.where(area >= TRK["min_area_km2"], 1.0, 0.0), np.nan)
ax.scatter(hh, dd, c=np.where(np.isnan(A), 0.85, np.where(A > 0, 0.25, 0.97)), cmap="Greys_r", vmin=0, vmax=1,
           marker="s", s=9, linewidths=0)
for _, e in EP.iterrows():
    col = plt.get_cmap("tab10")(e["episode"] % 10)
    for d0 in pd.date_range(e["start"].normalize(), e["end"].normalize()):
        a = max(e["start"], d0); b = min(e["end"], d0 + pd.Timedelta("23h45min"))
        y = (d0 - times[0].normalize()).days + 1
        ax.plot([a.hour + a.minute / 60, b.hour + b.minute / 60 + 0.25], [y + 0.32] * 2, color=col, lw=2.2, solid_capstyle="butt")
ax.set_ylim(len(days) + 0.7, 0.3); ax.set_xlim(0, 24.25)
ax.set_xticks(range(0, 25, 3)); ax.set_yticks(range(1, len(days) + 1, 2))
ax.set_xlabel("เวลา UTC (ชั่วโมง)"); ax.set_ylabel("วันที่ (พ.ค. 2567)")
ax.set_title("เหตุการณ์พายุ (แถบสี) บนช่วงที่มีพายุ (เทาเข้ม) — แถบที่ต่อไปบรรทัดถัดไป = คร่อม 00 UTC", loc="left", fontsize=8)
M.savefig(fig, "p3_episodes_calendar", outdir=OUT_ROOT / "figures")
plt.show()

## 3. ติดตามทีละเหตุการณ์ (ขนาน)

แต่ละ worker: โหลด CAPPI + P2 ช่วงเวลาของเหตุการณ์ (คร่อมหลายไฟล์ได้) → T-DaTing 15/10/5 นาที → ตารางเซลล์/เส้นทาง/footprint → เขียน label แยกตามวันเป็นไฟล์ส่วนย่อย → เก็บผลเป็น pickle (รันซ้ำแล้วข้ามได้) เริ่มจากเหตุการณ์ยาวที่สุดก่อนเพื่อกระจายงานให้สมดุล

In [ ]:
def ep_file(r):
    return EP_DIR / f"E{int(r['episode']):03d}_{r['start']:%Y%m%d%H%M}_{r['end']:%Y%m%d%H%M}.pkl"

def _work(r):
    f = ep_file(r)
    if f.exists() and not OVERWRITE:
        return int(r["episode"]), "skip", 0.0
    res = TR.run_episode(r, CAPPI_DIR, POLAR_DIR, PART_DIR, TERR, BEAM, TAGS, TRK["min_life_min"])
    with open(f, "wb") as fh:
        pickle.dump(res, fh)
    return int(r["episode"]), {t: len(v["tracks"]) for t, v in res["tags"].items()}, res["sec"]

todo = EP.sort_values("n_frames", ascending=False)
t_all = time.time()
with ProcessPoolExecutor(max(1, N_WORKERS // 2), mp_context=CTX) as ex:   # T-DaTing + ภาพ 5 นาทีใช้หน่วยความจำมาก
    for k, (ep, msg, sec) in enumerate(ex.map(_work, [r for _, r in todo.iterrows()]), 1):
        e = EP.loc[ep]
        print(f"[{k:3d}/{len(EP)}] E{ep:03d} {e['start']:%d %H:%M}–{e['end']:%d %H:%M} ({e['hours']:.1f} ชม.): {msg}"
              + (f" | {sec / 60:.1f} นาที" if sec else ""))
print(f"รวม {(time.time() - t_all) / 60:.1f} นาที")

In [ ]:
# รวมไฟล์ label ส่วนย่อยของทุกเหตุการณ์ → ไฟล์รายวัน KKN_YYYYMMDD_labels_<tag>.nc
for tag in TAGS:
    n = 0
    for d in days:
        if TR.merge_label_parts(PART_DIR, TRACK_DIR, d, tag, meta,
                                {"params": str(tt.PARAMS), "track_kw": str(tt.TRACK_KW), "tag": tag,
                                 "tracking": "episode-based (utils/tracking.py)"}) is not None:
            n += 1
    print(f"{tag}: รวมไฟล์ label {n} วัน")
print("ไฟล์ label รายวันใน", TRACK_DIR, ":", len(list(TRACK_DIR.glob("KKN_*_labels_*.nc"))))

## 4. รวมทั้งเดือน → ตาราง, footprint, กริดสรุป

In [ ]:
import geopandas as gpd
RES = []
for _, r in EP.iterrows():
    with open(ep_file(r), "rb") as fh:
        RES.append(pickle.load(fh))
phase_bins, phase_keys = [0, 9, 20, 31], [p["key"] for p in s["phases"]]
summary = []
for tag in TAGS:
    parts = [x["tags"][tag] for x in RES if tag in x["tags"]]
    if not parts:
        continue
    C = pd.concat([p["cells"] for p in parts], ignore_index=True)
    T = pd.concat([p["tracks"] for p in parts], ignore_index=True)
    F = pd.concat([p["footprints"] for p in parts], ignore_index=True)
    T["phase"] = pd.cut(T["day"].dt.day, phase_bins, labels=phase_keys)
    C.to_csv(DER / f"cells_{tag}.csv.gz", index=False, float_format="%.5g")
    T.to_csv(DER / f"tracks_{tag}.csv", index=False, float_format="%.5g")
    F = gpd.GeoDataFrame(F, geometry="geometry", crs=parts[0]["footprints"].crs)
    F["day"] = F["day"].astype(str)
    F.to_file(DER / f"footprints_{tag}.gpkg", driver="GPKG")
    life = T["lifetime_min"]
    summary.append({"tag": tag, "episodes": T["episode"].nunique(), "days": T["day"].nunique(), "tracks": len(T),
                    "life>=30min": int((life >= 30).sum()), "life>=60min": int((life >= 60).sum()),
                    "crosses_00utc": int(T["crosses_00utc"].sum()),
                    "censored": int((T["censored_start"] | T["censored_end"]).sum()), "max_life_min": life.max()})
summary = pd.DataFrame(summary); summary.to_csv(DER / "tracking_summary.csv", index=False)
summary

In [ ]:
n_obs = sum(v["n_obs"] for v in SCAN.values())
shape = meta["coverage"].shape
grids = {"n_obs_total": np.full(shape, n_obs, np.float32),
         "freq_ge35_pct": 100 * sum(v["freq35"] for v in SCAN.values()) / max(n_obs, 1),
         "freq_ge45_pct": 100 * sum(v["freq45"] for v in SCAN.values()) / max(n_obs, 1),
         "hail_col_count": sum(v["hail"] for v in SCAN.values()).astype(np.float32)}
for tag in TAGS:
    grids[f"track_density_{tag}"] = sum((x["tags"][tag]["density"] for x in RES if tag in x["tags"]), np.zeros(shape)).astype(np.float32)
    grids[f"origin_count_{tag}"] = sum((x["tags"][tag]["origin"] for x in RES if tag in x["tags"]), np.zeros(shape)).astype(np.float32)
for k in grids:
    grids[k] = np.where(meta["coverage"], grids[k], np.nan)
DIO.write_grids_2d(DER / "monthly_summary_1km.nc", grids, meta,
                   {"days": ",".join(days), "n_observed_volumes": n_obs, "n_episodes": len(EP),
                    "note": "freq = % of observed 15-min volumes; track density/origins count tracks with lifetime >= "
                            f"{TRK['min_life_min']} min (episode-based tracking); hail_col_count = storm-active volumes (P2)"})
print("บันทึก monthly_summary_1km.nc:", list(grids))

## 5. ตรวจสอบ

**(ก) เทียบกับ repo3 (3 พ.ค., เส้นทางที่เริ่ม 05–17 UTC):** repo3 = 114 เส้นทาง, ≥ 30 นาที 59 เส้นทาง (15 นาที) ต่างได้เล็กน้อยที่ขอบเวลา

**(ข) เส้นทางคร่อม 00 UTC:** ถ้าติดตามทีละวัน เส้นเหล่านี้จะถูกหั่นหรือหายไป — ดูว่ามีกี่เส้น อายุเท่าไร และอยู่ช่วงไหนของเดือน

In [ ]:
T15 = pd.read_csv(DER / "tracks_15min.csv", parse_dates=["day", "day_local", "start", "end"])
d3 = T15[(T15["day"] == "2024-05-03") & (T15["start"].dt.hour >= 5) & (T15["start"] <= "2024-05-03 17:00")]
print(f"(ก) 3 พ.ค. เริ่ม 05–17 UTC: {len(d3)} เส้นทาง, ≥ 30 นาที {(d3['lifetime_min'] >= 30).sum()} (repo3: 114 / 59)")
x = T15[T15["lifetime_min"] >= TRK["min_life_min"]].copy()
x["phase"] = pd.Categorical(x["phase"], [p["key"] for p in s["phases"]], ordered=True)   # เรียงตามเวลา ไม่ใช่ตัวอักษร
print(f"(ข) เส้นทาง ≥ {TRK['min_life_min']} นาที ที่คร่อม 00 UTC: {x['crosses_00utc'].sum()} จาก {len(x)} "
      f"({100 * x['crosses_00utc'].mean():.1f}%) | อายุมัธยฐาน คร่อม {x.loc[x['crosses_00utc'], 'lifetime_min'].median():.0f} "
      f"vs ไม่คร่อม {x.loc[~x['crosses_00utc'], 'lifetime_min'].median():.0f} นาที")
x.groupby("phase", observed=False)["crosses_00utc"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "คร่อม 00 UTC", "count": "ทั้งหมด", "mean": "สัดส่วน"}).round(3)

## 6. แผนที่เส้นทางทั้งเดือน (≥ 30 นาที, 15 นาที) ระบายสีตามช่วงของเดือน

In [ ]:
C15 = pd.read_csv(DER / "cells_15min.csv.gz", parse_dates=["day", "time"])
long_ = T15.loc[T15["lifetime_min"] >= TRK["min_life_min"]].set_index("uid")
fig, ax = plt.subplots(figsize=(7.2, 7.2))
dem_z, dem_ext = G.load_dem(s, build_if_missing=False)
M.base_map(ax, s, dem=(dem_z, dem_ext), dem_alpha=0.6)
for uid, g in C15[C15["uid"].isin(long_.index)].groupby("uid"):
    g = g.sort_values("time")
    ph = str(long_.loc[uid, "phase"])
    ax.plot(g["cx"] / 1e3, g["cy"] / 1e3, color=M.PHASE_COLORS[ph], lw=0.7, alpha=0.8, zorder=12)
    ax.plot(g["cx"].iloc[0] / 1e3, g["cy"].iloc[0] / 1e3, "o", ms=2, mfc="white", mec=M.PHASE_COLORS[ph], mew=0.6, zorder=13)
from matplotlib.lines import Line2D
ax.legend([Line2D([], [], color=c) for c in M.PHASE_COLORS.values()], [p["label_th"] for p in s["phases"]],
          loc="lower right", title=f"เส้นทาง ≥ {TRK['min_life_min']} นาที (○ = จุดเริ่ม)", title_fontsize=7)
ax.set_title(f"เส้นทางพายุ T-DaTing 15 นาที ทั้งเดือน (n = {len(long_)})", loc="left")
M.savefig(fig, "p3_tracks_month", outdir=OUT_ROOT / "figures")
plt.show()

**ขั้นต่อไป:** รัน P4 เพื่อจัดแพ็ก ตรวจ checksum คัดลอกวันหลักเข้า repo และเตรียมอัปโหลด Zenodo

## บันทึกผลเข้า Drive และ (เลือกได้) commit ขึ้น GitHub

`sync.push` เก็บไฟล์ที่จะเข้า repo ไว้ใน `KKN_derived_v1/repo_mirror/` notebook ถัดไป (เครื่องใหม่) จะดึงไปใช้เองตอนเริ่ม **รันเซลล์นี้ทุกครั้งก่อนปิด runtime**

ถ้าต้องการ commit ทันที ตั้ง `COMMIT = True` (ต้องมี secret `KKN_commit` = fine-grained token ของ repo นี้ สิทธิ์ Contents: Read and write) ตั้ง `dry_run=True` เพื่อดูรายการไฟล์ก่อน — ฟังก์ชันไม่พิมพ์ token ออกมา

In [ ]:
sync.push(OUT_ROOT)
COMMIT = False
if COMMIT:
    sync.git_push("P3: episode-based tracks, cells, footprints, monthly summary", dry_run=False)